In [ ]:
from src import *
from src.internals import *
from dataclasses import dataclass, field
import random
random.seed(42)


# Shaders, Materials, and Lights

In the previous notebooks, we focused mainly on geometry, rays, and intersections. That was enough to determine where a ray hits an object, but not yet what the object should look like. To produce a meaningful image, we also need to describe how surfaces respond to light.

This is where materials, shaders, and lights become important.

A **material** describes the visual properties of a surface. It can store information such as color, shininess, reflectivity, or transparency. In other words, the material tells us what kind of surface the object has.

A **shader** is responsible for computing the final color at a surface point. It uses the material properties together with information about the scene, such as the surface normal, the viewing direction, and the light sources. In other words, the shader tells us how the surface should be shaded.

A **light** represents a source of illumination in the scene. Without lights, most shaders would have no information about where illumination comes from, and objects would appear dark or would need to be drawn using only a constant color.

In this notebook, we will start with very simple examples. First, we will implement materials and shaders that return only a constant color, without considering any lighting. These simple models are useful because they help us verify that intersections, object assignment, and rendering all work correctly. They are also good for debugging, since they let us focus on one part of the rendering pipeline at a time.

After that, we can move to shaders that take lights into account. This will allow us to see how the same object can look different depending on the shading model and the lighting setup.

These first shaders are not physically based, but that is intentional. For an educational ray tracer, simple models are often the best starting point: they are easier to read, easier to implement, and easier to experiment with. Once the basic ideas are clear, more advanced and realistic shading models can be introduced later.

---
## Color Material and Shader

- Simple material that stores a base color and returns it when sampled.
- Simple shader that returns the base color of the material without any lighting calculations.
$$L = \text{base\_color}$$


In [ ]:
@dataclass
class ColorMaterial(Material):
    base_color: Color = field(default_factory=lambda: Color.custom_rgb(255, 255, 255)) # default to white color
    def get_color(self) -> Color:
        return self.base_color

    # this will be used when we want to sample the material for a given hit point
    def sample(self, hit: Vertex) -> Color:
        return self.base_color

class ColorShader(LocalShading):
    """
    A simple shader that returns the base color of the material without any lighting calculations. This can be useful for debugging or for rendering objects with a flat color.
    """
    def shade(self, hit: SurfaceInteraction, light: Light, view_dir: Vector, scene: Scene | None = None) -> Color:
        return hit.material.get_color()

    def shade_multiple_lights(self, hit: SurfaceInteraction, lights: list[Light], view_dir: Vector, scene: Scene | None = None) -> Color:
        """
        Shade ignoring multiple lights, since this shader is light independent and only returns the base color of the material.
        """
        return hit.material.get_color()

#### Visualize the scene with color shader
- We will create a simple scene with a few objects, assign them the ColorMaterial, and render the scene using the ColorShader.

In [ ]:
my_color_material = ColorMaterial(base_color=Color.custom_rgb(255, 0, 0))

red_sphere = Object(
    geometry=Sphere(),
    material=my_color_material
).scale(1.5, 2.0, 1.2).translate(0.0, 0.0, -4.0)

blue_box = Object(
    geometry=Box(),
    material=ColorMaterial(base_color=Color.custom_rgb(0, 0, 255))
).scale(1.0, 1.0, 1.0).translate(2.1, 0.0, -4.0)

green_box = Object(
    geometry=Box(),
    material=ColorMaterial(base_color=Color.custom_rgb(0, 255, 0))
).scale(1.0, 1.0, 1.0).translate(-2.1, 0.0, -3.0)

objects = [red_sphere, blue_box, green_box]

camera = PinholeCamera(
    origin=Vertex(0.0, 0.0, 0.0),
    direction=Vector(0.0, 0.0, -1.0),
    fov_deg=60.0
)

scene = Scene(objects=objects, lights=[], camera=camera, skybox="black") # this shaders is light independent, so we can render the scene without any lights

rt = LinearRenderLoop(
    render_config=RenderConfig(resolution=Resolution.R144p, samples_per_pixel=1, max_depth=1),
    preview_config=PreviewConfig(progress_display=ProgressDisplay.TQDM_IMAGE_PREVIEW),
    scene=scene,
    shading_model=ColorShader()
)

rt.render("images/color_shader.png")

---
## Normal Shader
We already used this one in the previous notebooks for debugging purposes, but now we will implement it as a proper shader. The normal shader is a simple shader that visualizes the normal vector at the hit point. It does this by mapping the normal vector components to RGB color values.

$$\text{colour} = \frac{\mathbf{n} + 1}{2} = \left(\frac{n_x+1}{2},\; \frac{n_y+1}{2},\; \frac{n_z+1}{2}\right)$$

This maps $[-1, +1]$ to $[0, 1]$. A normal pointing right $(+X)$ appears red, up $(+Y)$
green, toward the camera $(+Z)$ blue. If the sphere looks like a smooth colour gradient
across its surface, the normals are correct.

> **Note:** You can try to modify colors based on the normal vector components to create interesting visual effects.

In [ ]:
@dataclass
class NormalShader(LocalShading):
    """
    Simple object shader for previewing normals.
    """

    def shade(self, hit: SurfaceInteraction, light: Light | None, view_dir: Vector, scene: Scene | None = None) -> Color:
        """
        Shade based on the normal vector at the hit point.
        """
        material = hit.material
        n = hit.normal.normalize()

        # to change normal vector orientation we can use normal noise. This method allows us to change normal direction based on the noise value.
        noise = getattr(material, "normal_noise", None)
        norm = apply_noise_normal_perturbation(hit, noise, n)

        red = (norm.x + 1) * 0.5
        green = (norm.y + 1) * 0.5
        blue = (norm.z + 1) * 0.5

        return Color.linear_rgb(red, green, blue)

    def shade_multiple_lights(self, hit: SurfaceInteraction, lights: list[Light], view_dir: Vector, scene: Scene | None = None) -> Color:
        """
        Shade ignoring multiple lights normals are independent of lighting.
        """
        return self.shade(hit=hit, light=None, view_dir=view_dir)


#### Visualize the scene with normal shader
- We will use the same scene as before, but this time we will render it using the NormalShader to visualize the normals of the objects. Notice how the colors change based on the orientation of the surface normals, giving us insight into the geometry of the objects in the scene. Like the boxes have flat colors because they have flat normals, while the sphere has a smooth gradient because its normals change smoothly across its surface.

In [ ]:
normal_shader = NormalShader()
rt.change_shader(normal_shader)
rt.render("images/normal_shader.png")

---
## Depth Shader
The depth shader is another simple shader that visualizes the distance from the camera to the hit point. Closer objects will appear brighter, while farther objects will appear darker. This can be useful for debugging depth-related issues.

$$\text{colour} = 1.0 - \frac{\text{depth}}{\text{max\_depth}}$$

- we define a maximum depth value to control the range of distances that are visualized. Objects farther than this distance will be rendered as black, while objects closer will be rendered with varying shades of gray based on their distance from the camera.

> **Try it yourself:** You can experiment with different maximum depth values to see how it affects the visualization or change the color mapping to create different effects, such as using a color gradient instead of grayscale to represent depth. Or just try to change the 1.0 - (depth / self.max_depth) to different functions of depth to create interesting visual effects.

In [ ]:
@dataclass
class DepthShader(LocalShading):
    """
    Simple shader for previewing depth. This shader will render objects based on their distance from the camera, with closer objects appearing brighter and farther objects appearing darker.
    """
    max_depth: float = 10.0

    def shade(self, hit: SurfaceInteraction, light: Light | None, view_dir: Vector, scene: Scene | None = None) -> Color:
        """
        Shade based on the depth of the hit point from the camera. Closer objects will be brighter, while farther objects will be darker.
        """
        depth = min(hit.geom.dist, self.max_depth)
        intensity = 1.0 - (depth / self.max_depth)
        gray_value = int(intensity * 255)
        return Color.custom_rgb(gray_value, gray_value, gray_value)


    def shade_multiple_lights(self, hit: SurfaceInteraction, lights: list[Light], view_dir: Vector, scene: Scene | None = None) -> Color:
        """
        Shade ignoring multiple lights depth is independent of lighting.
        """
        return self.shade(hit=hit, light=None, view_dir=view_dir)

#### Visualize the scene with depth shader

In [ ]:
depth_shader = DepthShader(max_depth=4.5)
rt.change_shader(depth_shader)
rt.render("images/depth_shader.png")

---
### Dot Product Shader
The dot product shader is a simple shader that visualizes the angle between the surface normal and a camera or light direction. It computes the dot product between the normal vector and the view direction (or light direction) and maps it to a color. This can help us understand how the surface is oriented relative to the camera or light source. On the color is applied sine function to create a more interesting visual effect, but you can experiment and try to make your own one dot product shader.

> **Try it yourself:** Try to make your custom dot product shader that uses different color mapping.

#### Visualize the scene with dot product shader normal-camera

In [ ]:
light = PointLight(position=Vertex(0.0, 5.0, -3.0), intensity=1.0, color=Color.custom_rgb(255, 255, 255))
scene.add_lights([light])
dot_product_normal_camera = DotProductShader()
rt.change_shader(dot_product_normal_camera)
rt.render("images/dot_product_shader.png")

#### Visualize the scene with dot product shader normal-light
- this shader takes the first light from the scene and computes the dot product between the surface normal and the direction to that light.

In [ ]:
dot_product_normal_light = DotProductShader(use_light=True)
rt.change_shader(dot_product_normal_light)
rt.render("images/dot_product_shader_light.png")

---
## How to compare shaders?
In this library, you can easily combine shaders using the `DiffShader`, which takes two shaders as input and produces a new shader that visualizes the difference between them. This can be useful for comparing how different shading models affect the appearance of the same scene. The `DiffShader` can use different methods for combining the two shaders, such as stripes, checkerboard, circles, etc. This allows you to see the differences in a more visually distinct way.

$$L = \text{use\_mask}(L_a, L_b)$$

> **Try it yourself:** Try to change the hash method of the DiffShader to see how it affects the visualization of the differences between the two shaders. You can also try to create your own custom shader that combines the two shaders in a different way, such as by blending their colors or by using a different mathematical operation to combine their outputs.

In [ ]:
diff_shader = DiffShader(a = dot_product_normal_camera, b = dot_product_normal_light, mask_method=MaskMethod.STRIPES)
# MaskMethod try different methods for combining the two shaders, such as MaskMethod.:
#     CHECKER
#     CHECKED_LINES
#     STRIPES
#     CIRCLES
#     HALF_IMAGE

rt.change_shader(diff_shader)
rt.render("images/diff_shader.png")

---
# Lights and shadows
In the previous shaders, we ignored the presence of light sources in the scene. Now we will implement a simple shader that takes lights into account. This will allow us to see how the same object can look different depending on the shading model and the lighting setup.

- in this example, we will implement simple point light with constant intensity and ambient light. The point light will have a position and an intensity, color and the ambient light will have only intensity and color because it is not a directional light source and have constant intensity across the scene even in shadows. This is simplified model, but it is good to demonstrate the concept of light sources and shadows in ray tracing.



In [ ]:
@dataclass
class MyPointLight(Light):
    type: LightType = LightType.POINT
    color: Color = field(default_factory=lambda: Color.custom_rgb(255, 255, 255))
    intensity: float = 0.4

    def intensity_at(self, point: Vertex) -> float:
        return self.intensity

    def get_color_at(self, point: Vertex) -> Color:
        return self.color

class MyAmbientLight(Light):
    type: LightType = LightType.AMBIENT
    color: Color = field(default_factory=lambda: Color.custom_rgb(255, 255, 255))
    intensity: float = 0.2
    # color

    def intensity_at(self, point: Vertex) -> float:
        return self.intensity

    def get_color_at(self, point: Vertex) -> Color:
        return self.color

#### Visualize the scene with lights

In [ ]:
# visualize the scene with lights
point_light_1 = MyPointLight(position=Vertex(2, 3, 0), intensity=1.0)
point_light_2 = MyPointLight(position=Vertex(-2, 3, 2), intensity=2.0)
ambient_light = AmbientLight(intensity=0.2)
lights = [point_light_2, ambient_light] # we will use only one point light to see the effect of shadows more clearly, but you can experiment with different combinations of lights to see how they affect the scene

sphere = Object(
    geometry=Sphere(center=Vertex(0, 0.5, -0.5), radius=1.0),
    material=PhongMaterial(
        base_color=Color.custom_rgb(100, 100, 255),
        spec_color=Color.custom_rgb(255, 255, 255),
        ambient_color=Color.custom_rgb(200, 50, 255),
        shininess=32,
        transparency=0.1,
        reflectivity=0.1
    )
)


plane = Object(
    geometry=Plane(point=Vertex(0, -1, 0), normal=Vector(0, 1, 0)),
    material=PhongMaterial(
        base_color=Color.custom_rgb(200, 200, 200),
        spec_color=Color.custom_rgb(255, 255, 255),
        shininess=4,
        transparency=0.0,
        reflectivity=0.05
    )
)

objects = [sphere, plane]

camera = PinholeCamera(
    fov_deg=65,
    origin=Vertex(0, 1, 3),
    direction=Vector(0, -0.2, -1).normalize(),
)

vis = Visualizer()
vis.create_empty_scene(size=4.0, figsize=(12, 8), show_axes_labels=False, show_arrows=False, show_grid=True, show_axes=True, show_xyz_labels=True)
vis.visualize_objects(objects)
vis.plot_camera_position_and_orientation(camera, show_frustum=True, frustum_depth=8, show_camera_orientation=False)
vis.visualize_lights_positions(lights=[point_light_1,point_light_2])
vis.set_title("Scene with Light Source", fontsize=12)
vis.savefig("scene_with_light.png", dpi=300)
vis.show()

#### Visualize shadow rays
- Shadow rays are rays that are cast from the hit point towards the light sources to determine if the point is in shadow or not. If the shadow ray intersect object before reaching the light source, then the point is in shadow and should not receive direct illumination from that light. Visualizing shadow rays can help us understand how shadows are formed in the scene and how they affect the appearance of objects.

- This is simplified model of shadows and does not take into account soft shadows, area lights, or other more complex shadowing effects, but it is a good starting point for understanding the concept of shadows in ray tracing.

> **Try it yourself:** You can experiment with different light positions and see how the shadow rays change.

> **Try it yourself:** Try to change U V coordinates to see how the shadow rays change. So you can see how the position of the hit point on the sphere affects the shadow rays and the resulting shadows in the scene.

In [ ]:
vis.create_empty_scene(size=3.0, figsize=(12, 8), show_axes_labels=False, show_arrows=False, show_grid=True, show_axes=False, show_xyz_labels=False)
vis.plot_camera_position_and_orientation(camera, show_frustum=False, frustum_depth=5, show_camera_orientation=False, show_plane=True)
vis.visualize_objects([sphere])

u = -0.01
v = -0.34

vis.show_image_plane_point(camera, u, v)
vis.visualize_lights_positions([point_light_1,point_light_2])

ray = camera.make_ray(u, v)

hit_point : SurfaceInteraction= sphere.intersect(ray)


vis.visualize_ray(ray, length=5.0, color='magenta', opacity=0.3, ended_by_hit_point=hit_point)
vis.visualize_normal_at_hit_point(hit_point,length=1.2)
vis.visualize_closest_intersection(ray, objects=[sphere], intersection_opacity=0.9, intersection_size=60)
vis.show_shadow_rays(hit_point, [point_light_1, point_light_2], color=Color.custom_rgb(200, 200, 0), objects=[sphere])

vis.savefig("shadow_rays.pdf", dpi=300, show_legend=False)
vis.show(show_legend=True)

### Visualize shadow rays for multiple rays
- in this loop of random rays we will visualize shadow rays for multiple hit points on the sphere. We will see how the light interacts with the surface of the sphere and how the shadow rays are cast towards the light sources. This will give us a better understanding of how shadows are formed in the scene and how they affect the appearance of objects.

> **Try it yourself:** Add more objects to the scene and see how the shadow rays interact with them. And how shadows are formed.

In [ ]:
vis.create_empty_scene(size=3.0, figsize=(12, 8), show_axes_labels=False, show_arrows=False, show_grid=True, show_axes=True, show_xyz_labels=True)
vis.plot_camera_position_and_orientation(camera, show_frustum=True, frustum_depth=8, show_camera_orientation=False)
vis.visualize_lights_positions([point_light_1,point_light_2])
vis.visualize_objects([sphere])

for i in range(100):
    u = random.uniform(-1, 1)
    v = random.uniform(-1, 1)
    ray = camera.make_ray(u, v)
    hit_point : SurfaceInteraction= sphere.intersect(ray)

    vis.visualize_ray(ray, length=5.0, color='magenta', opacity=0.01, ended_by_hit_point=hit_point)
    vis.visualize_normal_at_hit_point(hit_point)
    vis.show_shadow_rays(hit_point, [point_light_1,point_light_2], color=Color.custom_rgb(200, 200, 0), objects=[sphere])
    vis.visualize_closest_intersection(ray, objects=[sphere], intersection_opacity=0.8)

vis.set_title("Shadow Rays", fontsize=12)
vis.savefig("camera_ray.png", dpi=300)
vis.show()

---
# Rendering equation and more advanced shaders

The foundation of physically based rendering is the **rendering equation**, introduced by Kajiya in 1986 but in our notebooks we will work with simplified version of it, which is sufficient for understanding the basic concepts of ray tracing and shading. The equation describes how light interacts with surfaces and how it contributes to the final color seen by the camera.

$$
L_o(x, \omega_o) =
L_e(x, \omega_o) +
\int_{\Omega} f_r(x, \omega_i, \omega_o)\, L_i(x, \omega_i)\, (\omega_i \cdot n)\, d\omega_i
$$

This equation states that the light leaving a point $x$ in direction $\omega_o$ is equal to the sum of emitted light and reflected incoming light.
- The term $L_o(x, \omega_o)$ denotes the outgoing radiance, which is the quantity we want to compute.
- The term $L_e(x, \omega_o)$ represents emitted radiance, such as light coming directly from a light source.
- The integral over the hemisphere $\Omega$ collects light arriving from all directions above the surface.
- The function $f_r(x, \omega_i, \omega_o)$ is the BRDF, which describes how the material reflects incoming light toward the outgoing direction.
- The factor $(\omega_i \cdot n)$ is the cosine term from Lambert's law, accounting for the angle between the incoming direction and the surface normal.

> **Note:** Whitted ray tracing does not solve this integral directly. Instead, it replaces it with a sum over a finite number of lights and a single perfect mirror reflection. Path tracing, in contrast, approximates the equation using Monte Carlo sampling. That is instead of integrating over all possible incoming directions, it randomly samples a few directions and averages the results to estimate the integral.

> **Note:** In our simplified version of the shading model, light can exceed 1 and colors can be outside the 0-255 range. But for easy educational purposes is important to keep the math simple.

---
# Lambert Shader
The Lambert shader is a simple shading model that assumes that the surface reflects light equally in all directions (diffuse reflection). The intensity of the reflected light is proportional to the cosine of the angle between the surface normal and the light direction, which is known as Lambert's cosine law. This results in a matte appearance, where the brightness of the surface depends on how directly it faces the light source.

$$L = \text{base\_color} \cdot \max(0, \mathbf{n} \cdot \mathbf{l}) \cdot I \cdot C$$

Where:
- $L$ is the outgoing radiance (the color we want to compute).
- $\text{base\_color}$ is the inherent color of the material.
- $\mathbf{n}$ is the surface normal at the hit point.
- $\mathbf{l}$ is the normalized vector pointing from the hit point to the light source.
- $I$ is the intensity of the light at the hit point, which can decrease with distance if we have lights with falloff.
- $C$ is the color of the light at the hit point.

We are also checking for shadows in this shader. If the point is in shadow with respect to a light source, we return black color for that light, which means that the point does not receive direct illumination from that light. This is done by casting a shadow ray from the hit point towards the light source as mentioned in the previous section. This can be done in the integrator, but for simplicity and to keep the shader self-contained and support multiple types of lights, we are doing it in the shader.

In [ ]:
class MyLambertShader(LocalShading):
    def shade(self, hit: SurfaceInteraction, light: Light, view_dir: Vector, scene: Scene | None = None ) -> Color:
        material = hit.material

        light_direction, light_distance = light_dir_dist(hit, light)
        if in_shadow(hit, light_direction, light_distance, scene=scene):
            return Color.custom_rgb(0, 0, 0)

        n = hit.normal.normalize()
        l = (light.position - hit.point).normalize()

        ndotl = max(0.0, n.dot(l))
        return material.get_color() * ndotl * light.intensity_at(hit.point) * light.get_color_at(hit.point)

    def shade_multiple_lights(self, hit: SurfaceInteraction, lights: list[Light], view_dir: Vector, scene: Scene | None = None) -> Color:

        color = Color.custom_rgb(0, 0, 0)
        for light in lights:
            color += self.shade(hit, light, view_dir, scene)

        return color

### Visualize the scene with Lambert shader

In [ ]:
scene = Scene(objects=[sphere], lights=lights, camera=camera, skybox="black")
render_configuration = RenderConfig(
    resolution=Resolution.R360p,
    samples_per_pixel=1,
    max_depth=1,
)

preview_configuration = PreviewConfig(
    progress_display=ProgressDisplay.TQDM_IMAGE_PREVIEW,
)

rt = LinearRenderLoop(scene=scene, shading_model=MyLambertShader(), render_config=render_configuration, preview_config=preview_configuration)
rendered_image = rt.render("lambert_diffusion.png")

---
# Blinn–Phong Shader
The Blinn–Phong shader is an extension of the Phong shading model that uses the **half-vector** between the light direction and the view direction to compute the specular highlight. In contrast, the original Phong model uses the reflection vector. This formulation is widely used because it is simple to implement, computationally efficient, and produces visually plausible specular highlights.

The Blinn–Phong model combines diffuse, specular, and optionally ambient components, which makes it suitable for representing a range of materials from matte to glossy surfaces. The diffuse term is commonly based on Lambert’s cosine law, while the specular term depends on the angle between the surface normal and the half-vector.

$$
L = \text{diffuse} + \text{specular} + \text{ambient}
$$

Where:
$$
  \text{diffuse} = \text{base\_color} \cdot \max(0, \mathbf{n} \cdot \mathbf{l}) \cdot I \cdot C
  $$
$$
  \text{specular} = \text{spec\_color} \cdot \max(0, \mathbf{n} \cdot \mathbf{h})^{\text{shininess}} \cdot I \cdot C
  $$

$$
    \text{ambient} = \text{ambient\_color} \cdot I \cdot C
    $$

$$
  \mathbf{h} = \frac{\mathbf{l} + \mathbf{v}}{\|\mathbf{l} + \mathbf{v}\|}
 $$

  is the half-vector between the normalized light direction $\mathbf{l}$ and the normalized view direction $\mathbf{v}$.

Here, $\mathbf{n}$ is the surface normal, $I$ is the light intensity at the shaded point, and $C$ is the color of the light.

The ambient component may be included either as a constant term or as a contribution from an ambient light source, depending on the specific implementation. In recursive ray tracing, it is often useful to include the ambient contribution only for the primary ray, so that it is not added repeatedly at every recursive bounce.

The surface normal can also be perturbed by a noise function to simulate small surface irregularities without increasing geometric complexity. This technique is related to bump mapping or normal perturbation. By modifying the normal used for shading, it is possible to create the appearance of roughness or fine surface detail while keeping the underlying geometry unchanged. Procedural noise and its use for normal perturbation will be discussed in a later notebook.

In [ ]:
class MyBlinnPhongShader(LocalShading):
    """
    Blinn–Phong shader is an extension of the Phong shading model that uses the half-vector between the light direction and the view direction to calculate the specular highlight. This can produce more visually appealing results, especially for shiny surfaces, compared to the original Phong model which uses the reflection vector. The Blinn–Phong shader is widely used in real-time rendering applications due to its efficiency and improved visual quality over the traditional Phong shader.
    """

    def shade(self, hit: SurfaceInteraction, light: Light, view_dir: Vector, scene: Scene | None = None) -> Color:

        material : PhongMaterial = hit.material

        light_direction, light_distance = light_dir_dist(hit, light)
        light_direction = light_direction.normalize()

        view_dir = view_dir.normalize()

        # This responsibility can be done in Whitted integrator for example, but we are doing it here to show how it works and to make the system more flexible, so you can use this shader in different contexts and with different types of lights without having to worry about shadows in the integrator with multiple lights.
        if in_shadow(hit, light_direction, light_distance, scene=scene):
            return Color.custom_rgb(0, 0, 0)

        normal = hit.normal.normalize()
        # We can change the normal direction based on the noise value to get different visual but this is optional, and you can experiment with it to see how it affects the final image.
        # In the last notebook we will see how to use noise to create different visual effects, but for now we will just use the original normal vector without any perturbation.
        #normal = apply_noise_normal_perturbation(normal, getattr(material, "normal_noise", None), normal)

        diffuse = self._diffuse(material, normal, light_direction)
        specular = self._specular(material, normal, light_direction, view_dir)

        return (diffuse + specular) * light.intensity_at(hit.point) * light.get_color_at(hit.point)

    def shade_multiple_lights(self, hit, lights, view_dir, scene=None) -> Color:
        material = hit.material
        accum = Color(0, 0, 0)

        for light in lights:
            if light.type == LightType.AMBIENT:
                accum += light.intensity_at(hit.geom.point) * material.get_color()
            else:
                accum += self.shade(hit, light, view_dir, scene=scene)

        return Color.clamp_color_255(accum)

    @staticmethod
    def _diffuse(m: PhongMaterial, n: Vector, l: Vector) -> Color:
        color = m.get_color()
        ndotl = max(0.0, n.dot(l))
        return color * ndotl

    @staticmethod
    def _specular(m: Material, n: Vector, to_light: Vector, to_cam: Vector) -> Color:
        h = (to_light + to_cam).normalize()
        ndoth = max(0.0, n.dot(h))
        spec_color = m.get_specular_color()
        shininess = m.get_shininess()
        return spec_color * (ndoth ** shininess)

In [ ]:
sphere = Object(
    geometry=Sphere(center=Vertex(0, 0.5, -0.5), radius=1.0),
    material=PhongMaterial(
        base_color=Color.custom_rgb(100, 100, 255),
        spec_color=Color.custom_rgb(255, 255, 255),
        ambient_color=Color.custom_rgb(200, 50, 255),
        shininess=128,
    )
)

camera = PinholeCamera(
    fov_deg=50,
    origin=Vertex(0, 1, 3),
    direction=Vector(0, -0.15, -1).normalize(),
)

scene = Scene(objects=[sphere], lights=lights, camera=camera, skybox="black")

render_configuration = RenderConfig(
    resolution=Resolution.R360p,
    samples_per_pixel=1,
    max_depth=1,
)

preview_configuration = PreviewConfig(
    progress_display=ProgressDisplay.TQDM_IMAGE_PREVIEW,
)

rt = LinearRenderLoop(scene=scene, shading_model=MyBlinnPhongShader(), render_config=render_configuration, preview_config=preview_configuration)
img = rt.render("images/rendered_scene_with_lights.png")

## Now separate the diffuse and specular components to see how they contribute to the final image.

#### Ambient Shader
- This shader will only calculate the ambient component of the Blinn-Phong shader.

In [ ]:
class AmbientShader(LocalShading):
    """
    Only ambient component of the Blinn-Phong shader.
    """
    def shade(self, hit: SurfaceInteraction, light: Light, view_dir: Vector, scene: Scene | None = None) -> Color:
        if light.type != LightType.AMBIENT:
            return Color.custom_rgb(0, 0, 0)

        material: PhongMaterial = hit.material
        base_color = getattr(material, "diff_color", material.get_color())
        return base_color * light.intensity_at(hit.point)

    def shade_multiple_lights(self, hit, lights, view_dir, scene=None) -> Color:
        accum = Color.custom_rgb(0, 0, 0)
        for light in lights:
            accum += self.shade(hit, light, view_dir, scene=scene)
        return Color.clamp_color_255(accum)

#### Diffuse Shader
- This shader will only calculate the diffuse component of the Blinn-Phong shader. Same as the lambert shader, but we will keep it separate to see how it contributes to the final image and you can experiment with it to see how it affects the final image.

In [ ]:
class DiffuseShader(LocalShading):
    """
    Only diffuse component of the Blinn-Phong shader.
    """
    def shade(self, hit: SurfaceInteraction, light: Light, view_dir: Vector, scene: Scene | None = None) -> Color:

        material: PhongMaterial = hit.material
        light_direction, light_distance = light_dir_dist(hit, light)
        light_direction = light_direction.normalize()
        normal = hit.normal

        if in_shadow(hit, light_direction, light_distance, scene=scene):
            return Color.custom_rgb(0, 0, 0)

        light_intensity = light.intensity_at(normal)
        if light_intensity <= 0.0:
            return Color.custom_rgb(0, 0, 0)

        color = material.get_color()
        ndotl = max(0.0, normal.dot(light_direction))
        return color * ndotl * light_intensity

    def shade_multiple_lights(self, hit, lights, view_dir, scene=None) -> Color:
        accum = Color.custom_rgb(0, 0, 0)
        for light in lights:
            if light.type == LightType.AMBIENT:
                continue
            accum += self.shade(hit, light, view_dir, scene=scene)
        return accum.clamp_01()

#### Specular Shader of Blinn-Phong
- This shader will only calculate the specular component of the Blinn-Phong shader. This will allow us to see how the specular highlights contribute to the final image and how they interact with the light sources and the view direction. You can experiment with different shininess values to see how it affects the size and intensity of the specular highlights, or with different specular colors to see how it affects the color of the highlights.

In [ ]:
class SpecularShaderBlinn(LocalShading):
    """
    Only specular component of the Blinn-Phong shader.
    """
    def shade(self, hit: SurfaceInteraction, light: Light, view_dir: Vector, scene: Scene | None = None) -> Color:
        if light.type == LightType.AMBIENT:
            return Color.custom_rgb(0, 0, 0)

        m: PhongMaterial = hit.material
        light_direction, light_distance = light_dir_dist(hit, light)
        light_direction = light_direction.normalize()
        view_dir = view_dir.normalize()

        if in_shadow(hit, light_direction, light_distance, scene=scene):
            return Color.custom_rgb(0, 0, 0)

        light_intensity = light.intensity_at(hit.point)
        if light_intensity <= 0.0:
            return Color.custom_rgb(0, 0, 0)

        h = (light_direction + view_dir).normalize()
        ndoth = max(0.0, hit.normal.dot(h))

        spec_color = m.get_specular_color()
        shininess = m.get_shininess()

        return spec_color * (ndoth ** shininess) * light_intensity

    def shade_multiple_lights(self, hit, lights, view_dir, scene=None) -> Color:
        accum = Color.custom_rgb(0, 0, 0)
        for light in lights:
            accum += self.shade(hit, light, view_dir, scene=scene)
        return Color.clamp_color_255(accum)

---
#### Visualize the scene with different shaders
- Now we can visualize the scene with different shaders to see how they contribute to the final image. We can start with the ambient shader, then the diffuse shader, and finally the specular shader. This will allow us to see how each component of the Blinn-Phong shader contributes to the final appearance of the objects in the scene. We can also compare the specular highlights produced by the Blinn-Phong shader and the original Phong shader to see the differences in their visual results.

#### Ambient component

In [ ]:
plane = Object(
    geometry=Plane(point=Vertex(0, -1, 0), normal=Vector(0, 1, 0)),
    material=PhongMaterial(
        base_color=Color.custom_rgb(100, 100, 100),
        spec_color=Color.custom_rgb(255, 255, 255),
        shininess=10,
    )
)
scene.add_objects(plane)


ambient_shader = AmbientShader()
rt.change_shader(ambient_shader)
rt.render("images/ambient_component.png")

#### Diffuse component

In [ ]:
diffuse_shader = DiffuseShader()
rt.change_shader(diffuse_shader)
rt.render("images/diffuse_component.png")

#### Specular component of Blinn-Phong

In [ ]:
specular_shader_blinn = SpecularShaderBlinn()
rt.change_shader(specular_shader_blinn)
rt.render("images/specular_component_blinn_phong.png")

#### All components together
Now we can visualize the full Blinn-Phong shader again to see how the ambient, diffuse and specular components combine together to create the final image. This will allow us to see how the different components interact with each other and how they contribute to the overall appearance of the objects in the scene.

In [ ]:
blinn_phong_shader = MyBlinnPhongShader()
rt.change_shader(blinn_phong_shader)
rt.render("images/blinn_phong_shader.png")

---
## Compare each component with the full shader using DiffShader
- Now we can use the DiffShader to compare each component of the Blinn-Phong shader with the full shader. This will allow us to see the differences between the full Blinn-Phong shader and each of its components in a more visually distinct way. By using different mask methods, we can see the differences in different patterns, such as stripes, checkerboard, circles, etc.

#### blinn-phong shader vs ambient component

In [ ]:
diff_shader = DiffShader(a=blinn_phong_shader, b=ambient_shader, mask_method=MaskMethod.STRIPES)
rt.change_shader(diff_shader)
rt.render("images/diff_spec_diff_shader.png")

#### Blinn–Phong shader vs. diffuse component
- As can be seen, the diffuse component can make the resulting color values greater than 1. In this educational and simplified shader, this is acceptable, since the purpose is to demonstrate the behavior of the shading model rather than to achieve physical correctness. In physically based rendering, however, this should not occur, because a surface should not reflect more light energy than it receives. This is one of the consequences of energy conservation, which is respected by physically based formulations derived from the rendering equation.

In [ ]:
diff_shader = DiffShader(a=blinn_phong_shader, b=diffuse_shader, mask_method=MaskMethod.STRIPES)
rt.change_shader(diff_shader)
rt.render("images/diff_ambient_diff_shader.png")

##### blinn-phong shader vs specular component

In [ ]:
diff_shader = DiffShader(a=blinn_phong_shader, b=specular_shader_blinn, mask_method=MaskMethod.STRIPES)
rt.change_shader(diff_shader)
rt.render("images/spec_ambient_diff_shader.png")

---


---
##### Specular Shader of original Phong
- This shader calculates only the specular component of the original Phong model. Compared to the Blinn-Phong shader, the original Phong model uses the reflection vector instead of the half-vector to calculate the specular highlight. This allows the results of the two shaders to be compared directly. For the same shininess value, Blinn-Phong specular highlights are generally broader, while the original Phong model produces smaller and sharper highlights. Blinn-Phong is also often considered slightly more efficient, because it uses the half-vector instead of explicitly computing the reflection vector.

$$L = \text{spec\_color} \cdot \max(0, \mathbf{v} \cdot \mathbf{r})^{\text{shininess}} \cdot I \cdot C$$

Where:
- $\mathbf{r}$ is the reflection vector, calculated as $\mathbf{r} = 2(\mathbf{n} \cdot \mathbf{l})\mathbf{n} - \mathbf{l}$,
- $\mathbf{n}$ is the surface normal,
- $\mathbf{l}$ is the normalized vector pointing from the hit point to the light source,
- $\mathbf{v}$ is the normalized vector pointing from the hit point to the camera,
- $\text{spec\_color}$ is the specular color of the material,
- $\text{shininess}$ controls the sharpness of the specular highlight,
- $I$ is the intensity of the light at the hit point,
- $C$ is the color of the light at the hit point.

In [ ]:
class SpecularShaderPhong(LocalShading):
    """
    Only specular component of the original Phong shader.
    """
    def shade(self, hit: SurfaceInteraction, light: Light, view_dir: Vector, scene: Scene | None = None) -> Color:
        if light.type == LightType.AMBIENT:
            return Color.custom_rgb(0, 0, 0)

        m: PhongMaterial = hit.material
        light_direction, light_distance = light_dir_dist(hit, light)
        light_direction = light_direction.normalize()
        view_dir = view_dir.normalize()

        if in_shadow(hit, light_direction, light_distance, scene=scene):
            return Color.custom_rgb(0, 0, 0)

        light_intensity = light.intensity_at(hit.point)
        if light_intensity <= 0.0:
            return Color.custom_rgb(0, 0, 0)

        # reflection vector for the original Phong model
        r = (2 * hit.normal.dot(light_direction) * hit.normal - light_direction).normalize()
        rdotv = max(0.0, r.dot(view_dir))

        spec_color = m.get_specular_color()
        shininess = m.get_shininess()

        return spec_color * (rdotv ** shininess) * light_intensity

    def shade_multiple_lights(self, hit, lights, view_dir, scene=None) -> Color:
        accum = Color.custom_rgb(0, 0, 0)
        for light in lights:
            accum += self.shade(hit, light, view_dir, scene=scene)
        return Color.clamp_color_255(accum)


---
## Compare specular component of blinn-phong vs original phong
> **note** With higher shininess values, the differences between the two models become more pronounced, with the original Phong model producing smaller and sharper highlights compared to the broader highlights of the Blinn-Phong model. The choice between the two models can depend on the specific visual effect desired and the performance requirements of the application.

In [ ]:
specular_shader_phong = SpecularShaderPhong()
rt.change_shader(specular_shader_phong)
rt.render("images/specular_component_phong.png")

In [ ]:
diff_shader = DiffShader(a=specular_shader_blinn, b=specular_shader_phong, mask_method=MaskMethod.STRIPES)
rt.change_shader(diff_shader)
rt.render("images/diff_spec_blinn_phong_phong.png")

---
## Compare different values

> **Try it yourself:** Try to change values of parameters of the Phong material, such as shininess, specular color, or base color, and see how it affects the rendered image. You can also experiment with different light positions, intensities and colors to see how they interact with the material properties and affect the final appearance of the objects in the scene.

In [ ]:
x = [-3.1, -1.5, 0.0, 1.5, 3.1]
shininess = [256, 64, 16, 4, 2]

shininess_test_spheres = [
    Object(
        geometry=Sphere(center=Vertex(x, 0.0, -2.0), radius=0.6),
        material=PhongMaterial(
            base_color=Color.custom_rgb(100, 100, 255),
            spec_color=Color.custom_rgb(255, 255, 255),
            ambient_color=Color.custom_rgb(30, 30, 60),
            shininess=s,
        )
    )
    for x, s in zip(x, shininess)
]

scene = Scene(objects=shininess_test_spheres, lights=lights, camera=camera, skybox="black")
render_configuration = RenderConfig(
    resolution=Resolution.custom(800, 400),
    samples_per_pixel=1,
    max_depth=1,
)
preview_configuration = PreviewConfig(
    progress_display=ProgressDisplay.TQDM_IMAGE_PREVIEW,
)
rt = LinearRenderLoop(scene=scene, shading_model=MyBlinnPhongShader(), render_config=render_configuration, preview_config=preview_configuration)
rt.render("images/blinn_phong_shininess_comparison.png")

# Summary:

## References
- Phong, B. T. (1975). Illumination for Computer Generated Pictures. *Communications of the ACM*, 18(6), 311–317.
- Blinn, J. F. (1977). Models of Light Reflection for Computer Synthesized Pictures. *SIGGRAPH '77*, 192–198.
- Whitted, T. (1980). An Improved Illumination Model for Shaded Display. *Communications of the ACM*, 23(6), 343–349.
- Kajiya, J. T. (1986). The Rendering Equation. *SIGGRAPH '86*, 143–150.